# **1) Initial configuration**

* In this section, we upload the input, import the necessary libraries, and read the input file.


<br>

### **Upload the input file**

* Run the cell below and upload the file using the button shown in the cell output.

* The input to the pipeline must be a file in TSV format.

* After uploading the input datasets, run the next cells.



In [ ]:
from google.colab import files
uploaded_file = files.upload()
maf_file_name = list(uploaded_file.keys())[0]


Saving dlbcl_integrated_mafs.tsv to dlbcl_integrated_mafs.tsv


<br>


### **Import libraries**



In [ ]:
!pip install kmapper==2.1.0
import kmapper as km

import os
import warnings
import pandas as pd
import numpy as np
import seaborn as sns

import networkx as nx

import sklearn as sk
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.mixture import BayesianGaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
from matplotlib.backends.backend_pdf import PdfPages

# Visual progress bar
from tqdm.notebook import tqdm




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 4.1 MB/s eta 0:00:00


<br>

### **Initial settings**

In [ ]:
# Ignore all warnings
warnings.filterwarnings("ignore")

# Create the output folders
file_name_no_ext = os.path.splitext(maf_file_name)[0]
output_folder = "Output/" + file_name_no_ext
os.makedirs(output_folder, exist_ok=True)

def print_title(title): print("\n", title, "\n", "="*len(title), sep="")


<br>

### **Create folders**

In [ ]:
# Output folders
dirs = {
    "datasets": "Datasets"
}

# Create the output folders for path in dirs.values():
for path in dirs.values():
    os.makedirs(path, exist_ok=True)
    print(f"Directory '{path}' created or already exists.")


Directory 'Datasets' created or already exists.


<br>

### **Print MAF information**


In [ ]:
def print_title(title): print("\n", title, "\n", "="*len(title), sep="")

def print_inf_maf_data(maf_df):
    # Obtaining all unique tumor samples from MAF
    num_unique_samples = maf_df["Tumor_Sample_Barcode"].unique()

    # Obtaining all unique mutations from MAF
    num_unique_genes = maf_df["Hugo_Symbol"].unique()

    # Create the DataFrame with the number of mutations in each tumor sample
    number_mutations_df = maf_df.groupby("Tumor_Sample_Barcode").size().reset_index(name="Number of mutations")
    number_mutations_df = number_mutations_df.rename(columns={"Tumor_Sample_Barcode": "Sample"})

    # Calculate the mean and median of the number of mutations per sample
    mean_mutations = number_mutations_df["Number of mutations"].mean()
    median_mutations = number_mutations_df["Number of mutations"].median()

    # Print the results
    print_title("Information from the MAF data:")
    print(f"   Total mutations: {len(maf_df)}")
    print(f"   Number of unique tumor samples: {len(num_unique_samples)}")
    print(f"   Number of unique genes: {len(num_unique_genes)}")
    print(f"   Mean number of mutations per sample: {mean_mutations:.0f}")
    print(f"   Median number of mutations per sample: {median_mutations:.0f}")


<br>

### **Load files**

* The files with driver genes are automatically downloaded from the repository.

In [ ]:
# Load the canonical driver genes file
canonical_drivers_file_name = "ncg_canonical_cancer_drivers.csv"
!wget -O {dirs["datasets"] + "/" + canonical_drivers_file_name} https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/ncg_canonical_cancer_drivers.csv

# Load the customized gene panel file
customized_gene_panel_file_name = "customized_gene_panel-Identification_somatic_variants_in_ctDNA.csv"
!wget -O {dirs["datasets"] + "/" + customized_gene_panel_file_name} https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/customized_gene_panel-Identification_somatic_variants_in_ctDNA.csv

# Load the candidate cancer drivers file
candidate_drivers_file_name = "ncg_candidate_drivers_updated.csv"
!wget -O {dirs["datasets"] + "/" + candidate_drivers_file_name} https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/ncg_candidate_drivers_updated.csv

# Load file with oncogene (OG) and tumor suppressor (TSG) information.
og_tsg_file_name = "ncg_cancerdrivers_annotation_supporting_evidence.tsv"
!wget -O {dirs["datasets"] + "/" + og_tsg_file_name} https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/ncg_cancerdrivers_annotation_supporting_evidence.tsv

# Load file with the DLBCL gene network of the paper "Genetic and Functional Drivers of Diffuse Large B Cell Lymphoma" (Reddy, 2017)
dlbcl_gene_network_file_name = "dlbcl_gene_network-reddy2017.tsv"
!wget -O {dirs["datasets"] + "/" + dlbcl_gene_network_file_name} https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/dlbcl_gene_network-reddy2017.tsv

# Load file with the DLBCL genetic subtypes of the paper "A probabilistic classification tool for genetic subtypes of diffuse large B cell lymphoma with therapeutic implications" (Wright, 2020)
dlbcl_genetic_subtypes_file_name = "dlbcl_genetic_subtypes-wright2020.tsv"
!wget -O {dirs["datasets"] + "/" + dlbcl_genetic_subtypes_file_name} https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/dlbcl_genetic_subtypes-wright2020.tsv



--2026-06-02 20:59:48--  https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/ncg_canonical_cancer_drivers.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3437 (3.4K) [text/plain]
Saving to: ‘Datasets/ncg_canonical_cancer_drivers.csv’

Datasets/ncg_canoni 100%[===================>]   3.36K  --.-KB/s    in 0s      

2026-06-02 20:59:48 (29.2 MB/s) - ‘Datasets/ncg_canonical_cancer_drivers.csv’ saved [3437/3437]

--2026-06-02 20:59:48--  https://raw.githubusercontent.com/paulo-ribeiro/datasets/refs/heads/main/customized_gene_panel-Identification_somatic_variants_in_ctDNA.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (ra

<br>

<br><br><br>
<br><br><br>

---


# **1) Read files**



### **Read the MAF file**


In [ ]:
# MAF columns that will be in the dataframe
maf_columns = ["Hugo_Symbol", "Variant_Classification", "Variant_Type", "Tumor_Sample_Barcode"]

# Create the dataframe ignoring comment lines
maf_df = pd.read_csv(maf_file_name, sep="\t", usecols=maf_columns, comment="#")

print_inf_maf_data(maf_df)
print("\n\n")
display(maf_df)



Information from the MAF data:
   Total mutations: 62128
   Number of unique tumor samples: 1636
   Number of unique genes: 13585
   Mean number of mutations per sample: 38
   Median number of mutations per sample: 8





,Hugo_Symbol,Variant_Classification,Variant_Type,Tumor_Sample_Barcode
0,C8orf4,Missense_Mutation,SNP,DLBCL-DFCI_DLBCL_Goe05
1,SLC27A1,Nonsense_Mutation,SNP,DLBCL-DFCI_DLBCL_Goe05
2,HS3ST1,Missense_Mutation,SNP,DLBCL-DFCI_DLBCL_Goe05
3,FAF2,Nonsense_Mutation,SNP,DLBCL-DFCI_DLBCL_Goe05
4,PTPRC,Silent,SNP,DLBCL-DFCI_DLBCL_Goe05
...,...,...,...,...
62123,ZBTB43,3'Flank,SNP,TCGA-VB-A8QN-01
62124,RAB9A,3'UTR,SNP,TCGA-VB-A8QN-01
62125,IGLJ3,5'UTR,DEL,TCGA-FF-8042-01
62126,IGKJ2,In_Frame_Del,DEL,TCGA-FF-8042-01


<br>


### **Customized gene panel**


In [ ]:
customized_gene_panel_df = pd.read_csv(dirs["datasets"] + "/" + customized_gene_panel_file_name).iloc[:,0].to_list()
print(f"   Number of genes in the customized panel: {len(customized_gene_panel_df)}")


   Number of genes in the customized panel: 238


<br>


### **Canonical drivers**


In [ ]:
canonical_cancer_drivers = pd.read_csv(dirs["datasets"] + "/" + canonical_drivers_file_name).iloc[:,0].to_list()
print(f"   Number of canonical cancer drivers genes: {len(canonical_cancer_drivers)}")


   Number of canonical cancer drivers genes: 591


<br>


### **Candidate drivers**

In [ ]:
candidate_cancer_drivers = pd.read_csv(dirs["datasets"] + "/" + candidate_drivers_file_name).iloc[:,0].to_list()
print(f"   Number of candidate cancer drivers genes: {len(candidate_cancer_drivers)}")


   Number of candidate cancer drivers genes: 2756


<br>


### **OG and TSG**

In [ ]:
og_tsg_genes = pd.read_csv(dirs["datasets"] + "/" + og_tsg_file_name, sep="\t", usecols=["symbol", "NCG_oncogene", "NCG_tsg"])

# Generate the list of unique oncogenes
oncogene_list = og_tsg_genes[og_tsg_genes["NCG_oncogene"] == 1]["symbol"].unique().tolist()

# Generate the list of unique tumor suppressor genes
tsg_list = og_tsg_genes[og_tsg_genes["NCG_tsg"] == 1]["symbol"].unique().tolist()

print(f"Number of unique Oncogenes (OG): {len(oncogene_list)}")
print(f"Number of unique Tumor Suppressor Genes (TSG): {len(tsg_list)}")


Number of unique Oncogenes (OG): 256
Number of unique Tumor Suppressor Genes (TSG): 254


<br>


### **DLBCL gene network**

In [ ]:
# Read file with the DLBCL gene network of the paper "Genetic and Functional Drivers of Diffuse Large B Cell Lymphoma" (Reddy, 2017)
dlbcl_gene_network_df = pd.read_csv(dirs["datasets"] + "/" + dlbcl_gene_network_file_name, sep="\t")

# Formatting of the DLBCL gene network data
dlbcl_gene_network_df["EdgeType"] = dlbcl_gene_network_df["EdgeType"].replace({
    "exclusion": "Mutual exclusivity",
    "overlap": "Co-occurrence"
})

num_genes = pd.concat([dlbcl_gene_network_df["Gene1"], dlbcl_gene_network_df["Gene2"]]).nunique()
print(f"   Number of nodes in the network: {num_genes}")
print(f"   Number of edges in the network: {len(dlbcl_gene_network_df)}")


   Number of nodes in the network: 54
   Number of edges in the network: 121


<br>


### **DLBCL subtypes**

In [ ]:
# Read file with the DLBCL genetic subtypes of the paper "A probabilistic classification tool for genetic subtypes of diffuse large B cell lymphoma with therapeutic implications" (Wright, 2020)
dlbcl_genetic_subtypes_df = pd.read_csv(dirs["datasets"] + "/" + dlbcl_genetic_subtypes_file_name, sep="\t")
#dlbcl_genetic_subtypes_df = pd.read_csv("DLBCL_subtypes_artigo2017.tsv", sep="\t")
#dlbcl_genetic_subtypes_df = pd.read_csv("DLBCL_subtypes_artigo2020.tsv", sep="\t")


print(f"   Number of genes in the file: {len(dlbcl_genetic_subtypes_df["Gene"].unique())}")
print(f"   Number of subtypes in the file: {len(dlbcl_genetic_subtypes_df["Subtype"].unique())}")



   Number of genes in the file: 165
   Number of subtypes in the file: 6


<br>

<br><br><br>

---


# **2) Data preprocessing**

<br>

### **Filter mutations**

  - Filtering MAF mutations according to the following criteria:

    - Removal of mutations with missing values in critical columns.

    - Selection of only non-silent mutations with potential functional impact.


In [ ]:
#  Remove rows with missing values in critical columns
critical_columns = ["Hugo_Symbol"]
maf_df.dropna(subset=critical_columns, inplace=True)

# Filter for non-silent mutations
non_silent_mutations = [
    "Frame_Shift_Del", "Frame_Shift_Ins", "In_Frame_Del", "In_Frame_Ins",
    "Missense_Mutation", "Nonsense_Mutation", "Nonstop_Mutation",
    "Splice_Site", "Translation_Start_Site"
]
maf_df = maf_df[maf_df["Variant_Classification"].isin(non_silent_mutations)]

print_inf_maf_data(maf_df)



Information from the MAF data:
   Total mutations: 48409
   Number of unique tumor samples: 1634
   Number of unique genes: 11412
   Mean number of mutations per sample: 30
   Median number of mutations per sample: 8


<br>

### **Filter hypermutated samples**

  - Hypermutated samples should be excluded as they are often noisy outliers, which can distort analyses of clustering methods.


In [ ]:
# Count mutations per sample
mutations_per_sample = maf_df["Tumor_Sample_Barcode"].value_counts()

# Calculate Q1, Q3 and IQR
Q1 = mutations_per_sample.quantile(0.25)
Q3 = mutations_per_sample.quantile(0.75)
IQR = Q3 - Q1
threshold = min(Q3 + 4.5 * IQR, 600)

# Remove hypermutated samples from the dataframe
hypermutated_samples = mutations_per_sample[mutations_per_sample > threshold].index.tolist()
maf_df = maf_df[~maf_df["Tumor_Sample_Barcode"].isin(hypermutated_samples)]

print_inf_maf_data(maf_df)



Information from the MAF data:
   Total mutations: 17384
   Number of unique tumor samples: 1487
   Number of unique genes: 3651
   Mean number of mutations per sample: 12
   Median number of mutations per sample: 7


<br>

### **Filter samples with few mutations**

  - Removing tumor samples with an insufficient number of mutations.



In [ ]:
# Count mutations per sample
mutations_per_sample = maf_df["Tumor_Sample_Barcode"].value_counts()

# Set the minimum threshold of mutations per sample
min_mutations = 5

# Filter samples that have enough mutations
valid_samples = mutations_per_sample[mutations_per_sample >= min_mutations].index.tolist()
maf_df = maf_df[maf_df["Tumor_Sample_Barcode"].isin(valid_samples)]

print_inf_maf_data(maf_df)



Information from the MAF data:
   Total mutations: 16295
   Number of unique tumor samples: 1095
   Number of unique genes: 3641
   Mean number of mutations per sample: 15
   Median number of mutations per sample: 10


<br>

### **Filter key genes**

* Filter genes from the customized panel.
* Filter the most frequently mutated genes.

In [ ]:
# Filter dataframe to keep only genes of the customized gene panel
panel_maf_df = maf_df[maf_df['Hugo_Symbol'].isin(customized_gene_panel_df)].copy()

mutations_counts = (
    panel_maf_df.groupby("Hugo_Symbol")
          .size()
          .reset_index(name="count")
          .sort_values("count", ascending=False)
)

# Filter genes present in more samples
top_genes = mutations_counts.head(30)["Hugo_Symbol"].tolist()
top_genes_maf_df = panel_maf_df[panel_maf_df["Hugo_Symbol"].isin(top_genes)].copy().reset_index()
top_genes_maf_df["VAF"] = 1

print_inf_maf_data(panel_maf_df)



Information from the MAF data:
   Total mutations: 8900
   Number of unique tumor samples: 1090
   Number of unique genes: 173
   Mean number of mutations per sample: 8
   Median number of mutations per sample: 7


<br>

### **Create binary data**

   - **Columns**: most frequent genes in patients
   - **Rows**: patients
   - **Cells**: VAF of mutations

In [ ]:
# Filter dataset columns for the causal discovery algorithm
vaf_mutations_df = top_genes_maf_df[["Hugo_Symbol", "Tumor_Sample_Barcode", "VAF"]]

# Create the dataset in the format required for the causal discovery algorithm
data_df = vaf_mutations_df.pivot_table(index="Tumor_Sample_Barcode",
                                                 columns="Hugo_Symbol",
                                                 values="VAF",
                                                 aggfunc="mean")

# Convert dataset to binary: 1 if mutation exists (value is not NaN), 0 otherwise
binary_data_df = data_df.notna().astype(int)

# Create matrix with data for causal discovery algorithm
data_matrix = binary_data_df.to_numpy()

# List of gene names for the causal discovery algorithm
genes_labels = binary_data_df.columns.tolist()

display(binary_data_df)


Hugo_Symbol,ARID1A,ARID1B,B2M,BCL2,BTG1,CARD11,CD79B,CREBBP,DUSP2,EP300,...,SETD1B,SGK1,SMARCA4,SOCS1,SPEN,TBL1XR1,TET2,TNFAIP3,TNFRSF14,TP53
Tumor_Sample_Barcode,,,,,,,,,,,,,,,,,,,,,
DFCI_DLBCL_Goe05,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
DLBCL-DFCI_DLBCL_Goe05,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
DLBCL-DFCI_DLBCL_Goe16,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
DLBCL-LS1304,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DLBCL-LS1395,0,1,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-FF-8046-01,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-FF-A7CW-01,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
TCGA-GR-A4D6-01,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


<br>

### **Data formatting**

* The dataset to be used by the causal discovery algorithms has the following structure:

   - **Columns**: most frequent genes in patients
   - **Rows**: patients
   - **Cells**: VAF of mutations

In [ ]:
data_df = binary_data_df.transpose()

# Order the genes with mutations in the largest number of samples.
sample_count = data_df.sum(axis=1)
data_df = data_df.loc[sample_count.sort_values(ascending=False).index]

# Save dataframe to TSV file
data_df.to_csv("data_df.tsv", sep="\t", index=False)

display(data_df)


Tumor_Sample_Barcode,DFCI_DLBCL_Goe05,DLBCL-DFCI_DLBCL_Goe05,DLBCL-DFCI_DLBCL_Goe16,DLBCL-LS1304,DLBCL-LS1395,DLBCL-LS148,DLBCL-LS155,DLBCL-LS1583,DLBCL-LS1683,DLBCL-LS2089,...,RG115,RG132,RG135,RG136,TCGA-FA-A6HO-01,TCGA-FF-8046-01,TCGA-FF-A7CW-01,TCGA-GR-A4D6-01,TCGA-GS-A9TU-01,TCGA-GS-A9TX-01
Hugo_Symbol,,,,,,,,,,,,,,,,,,,,,
PIM1,1,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
MYD88,1,1,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
TP53,1,1,0,0,1,0,0,0,1,1,...,0,1,0,0,0,0,0,0,0,0
CREBBP,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
HIST1H1E,0,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,1,0,0,0
BCL2,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SOCS1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
TNFRSF14,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
SETD1B,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


<br><br><br>
<br><br><br>

---


# **3) Functions - Mapper**

  - In this section, functions related to the Mapper algorithm are implemented.

<br>

### **Functions - Metrics and scores calculations**

* Functions to calculate metrics and scores from a Mapper algorithm result.

In [ ]:
# Calculate connected components

def calc_connected_components(mapper_graph, num_clusters):
    num_connected_components = 0
    if num_clusters > 0:
        # Create a NetworkX graph from Mapper output
        G = nx.Graph()

        # Add nodes (clusters) to the graph
        G.add_nodes_from(mapper_graph["nodes"].keys())

        # Add the edges
        for source, targets in mapper_graph['links'].items():
            for target in targets:
                G.add_edge(source, target)

        # Calculate the number of connected components
        num_connected_components = nx.number_connected_components(G)

    return num_connected_components


<br>

In [ ]:
# Calculate the number of unclustered samples

def calc_unclustered_samples(mapper_graph, total_samples, num_clusters):
    if num_clusters == 0:
        # Handle the case where no clusters are formed
        unclustered_samples = total_samples
    else:
        samples_in_clusters = set()
        for node_name in mapper_graph["nodes"]:
            member_indices = mapper_graph["nodes"][node_name]
            samples_in_clusters.update(member_indices)
        unclustered_samples = total_samples - len(samples_in_clusters)

    return unclustered_samples


<br>

In [ ]:
# Calculate the score of a metric

def calc_score(current_value, range_dict):
    vmin = range_dict["vmin"]
    vmax = range_dict["vmax"]
    range_width = vmax - vmin

    if current_value < vmin:
        distance = vmin - current_value
    elif current_value > vmax:
        distance = current_value - vmax
    else:
        distance = 0

    if range_width == 0:
        score = 1.0 if distance == 0 else 0.0
    else:
        score = 1 - (distance / range_width)

    return score


<br>

In [ ]:
# Calculate the combined score of a Mapper algorithm result

def calc_combined_score(score_params, unclustered_samples, num_clusters, num_edges,
                        num_connected_components, std_dev_cluster_sizes):

    score1 = calc_score(unclustered_samples, score_params["target_unclustered"])
    score2 = calc_score(num_clusters, score_params["target_clusters"])
    score3 = calc_score(num_edges, score_params["target_edges"])
    score4 = calc_score(num_connected_components, score_params["target_connected_components"])
    score5 = calc_score(std_dev_cluster_sizes, score_params["target_cluster_size_variation"])

    total_weights = sum(score_params[f"weight_score{i}"] for i in range(1, 6))

    combined_score = (score_params["weight_score1"] * score1 +
                      score_params["weight_score2"] * score2 +
                      score_params["weight_score3"] * score3 +
                      score_params["weight_score4"] * score4 +
                      score_params["weight_score5"] * score5 ) / total_weights
    combined_score = max(0, combined_score)

    return combined_score


<br>

### **Function - Format plot titles**

* This function formats the title of the plots.

In [ ]:
# Format the projection function title
def format_projection_title(proj_func):
    if isinstance(proj_func, PCA):
        projection_title = (f"Projection: PCA (n_components={proj_func.n_components})")
    elif isinstance(proj_func, TSNE):
        projection_title = (f"Projection: t-SNE (n_components={proj_func.n_components})")
    else:
        projection_title = (f"Projection: {proj_func}")

    return (projection_title)

In [ ]:
# Format the clustering method title
def format_clustering_title(clust_alg):
    if isinstance(clust_alg, sk.cluster.DBSCAN):
        clustering_title = (f"Clustering: DBSCAN (eps={clust_alg.eps})")
    elif isinstance(clust_alg, sk.mixture.BayesianGaussianMixture):
        clustering_title = (f"Clustering: Bayesian Gaussian Mixture (n_components={clust_alg.n_components})")
    elif isinstance(clust_alg, sk.cluster.AgglomerativeClustering):
        clustering_title = (f"Clustering: Hierarchical Agglomerative Clustering (distance_threshold={clust_alg.distance_threshold})")
    else:
        clustering_title = (f"Clustering: {clust_alg}")

    return (clustering_title)

In [ ]:
# Format the title with the projection function name and grouping method
def format_plot_title(proj_func, clust_alg):
    projection_title = format_projection_title(proj_func)
    clustering_title = format_clustering_title(clust_alg)
    return (projection_title + "\n\n" + clustering_title)


<br>

### **Function - Run Mapper analysis**

* Function that performs a parameter sweep for the Kepler Mapper algorithm.

In [ ]:
def run_mapper_analysis(mapper, normalized_data, projected_data, proj_func,
                        clust_alg, n_cubes_range, perc_overlap_range):
    results_data = []

    # Loop to run Mapper and collect metrics
    for n_cubes in n_cubes_range:
        for perc_overlap in perc_overlap_range:
            try:
                # Run the Mapper algorithm with the parameters of the current iteration
                cover = km.Cover(n_cubes=n_cubes, perc_overlap=perc_overlap)
                mapper_graph = mapper.map(lens=projected_data, X=normalized_data,
                              clusterer=clust_alg, cover=cover)

                # Calculate metrics
                num_clusters = len(mapper_graph["nodes"])
                num_edges = sum(len(v) for v in mapper_graph['links'].values())
                num_connected_components = calc_connected_components(mapper_graph, num_clusters)
                node_sizes = [len(members) for members in mapper_graph["nodes"].values()]
                std_dev_cluster_sizes = np.std(node_sizes) if num_clusters > 0 else 0.0
                unclustered_samples = calc_unclustered_samples(mapper_graph, total_samples, num_clusters)

                max_cluster_size = max(node_sizes) if num_clusters > 0 else 0
                num_large_clusters = len([size for size in node_sizes if size >= 15])
                num_small_clusters = len([size for size in node_sizes if size < 3])

                # Calculate the combined score of a Mapper algorithm result
                combined_score = calc_combined_score(score_params, unclustered_samples,
                                  num_clusters, num_edges, num_connected_components, std_dev_cluster_sizes)

                # Store the results
                results_data.append({
                  "projection_function": str(proj_func),
                  "clustering_algorithm": str(clust_alg),
                  "n_cubes": n_cubes,
                  "perc_overlap": perc_overlap,
                  "unclustered_samples": unclustered_samples,
                  "num_clusters": num_clusters,
                  "num_edges": num_edges,
                  "num_connected_components": num_connected_components,
                  "std_dev_cluster_sizes": std_dev_cluster_sizes,
                  "max_cluster_size": max_cluster_size,
                  "num_large_clusters": num_large_clusters,
                  "num_small_clusters": num_small_clusters,
                  "combined_score": combined_score
                })
            except ValueError as e:
                print(f"    Warning: Skipping combination for clusterer '{clust_alg}' with n_cubes={n_cubes}, perc_overlap={perc_overlap}. Reason: {e}")
                continue

    results_df = pd.DataFrame(results_data)
    return results_df


<br>

### **Function - Heatmaps of metrics**

* Function that generates heatmaps of metrics from the results of the Mapper algorithm.

* The heatmaps generated are for the following metrics:

  1. Number of unclustered samples

  2. Number of clusters

  3. Number of edges

  4. Number of connected components

  5. Standard deviation of cluster sizes

  6. Combined score

In [ ]:
def plot_heatmaps(results_df, pdf_pages, title=None):

    if results_df.empty:
        print("No results were generated. Skipping heatmap generation.")
        return

    # Create pivot tables for each metric, which is the expected format for heatmaps
    heatmap_unclustered = results_df.pivot(index="perc_overlap", columns="n_cubes", values="unclustered_samples")
    heatmap_num_clusters = results_df.pivot(index="perc_overlap", columns="n_cubes", values="num_clusters")
    heatmap_num_edges = results_df.pivot(index="perc_overlap", columns="n_cubes", values="num_edges")
    heatmap_std_dev_cluster_sizes = results_df.pivot(index="perc_overlap", columns="n_cubes", values="std_dev_cluster_sizes")
    heatmap_connected_components = results_df.pivot(index="perc_overlap", columns="n_cubes", values="num_connected_components")
    heatmap_combined_score = results_df.pivot(index="perc_overlap", columns="n_cubes", values="combined_score")

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes = axes.flatten()
    fig.suptitle(title, fontsize=14)

    # Heatmap 1: Unclustered samples
    sns.heatmap(heatmap_unclustered, ax=axes[0], cmap="coolwarm", annot=True, fmt=".0f", linewidths=.5, vmin=0, vmax=10)
    axes[0].set_title("Unclustered Samples")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Overlap")

    # Heatmap 2: Number of Clusters
    sns.heatmap(heatmap_num_clusters, ax=axes[1], cmap="viridis", annot=True, fmt=".0f", linewidths=.5, vmin=0, vmax=10)
    axes[1].set_title("Number of Clusters")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("")

    # Heatmap 3: Number of Edges
    sns.heatmap(heatmap_num_edges, ax=axes[2], cmap="viridis", annot=True, fmt=".0f", linewidths=.5, vmin=0, vmax=10)
    axes[2].set_title("Number of Edges")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("")

    # Heatmap 4: Connected components
    sns.heatmap(heatmap_connected_components, ax=axes[3], cmap="viridis", annot=True, fmt=".0f", linewidths=.5, vmin=0, vmax=10)
    axes[3].set_title("Connected Components")
    axes[3].set_xlabel("Intervals")
    axes[3].set_ylabel("Overlap")

    # Heatmap 5: Standard Deviation of Cluster Sizes
    sns.heatmap(heatmap_std_dev_cluster_sizes, ax=axes[4], cmap="viridis", annot=True, fmt=".1f", linewidths=.5, vmin=0.0, vmax=10)
    axes[4].set_title("Cluster Size Variation")
    axes[4].set_xlabel("Intervals")
    axes[4].set_ylabel("")

    # Heatmap 6: Combined Score
    sns.heatmap(heatmap_combined_score, ax=axes[5], cmap="coolwarm_r", annot=True, fmt=".2f", linewidths=.5, vmin=0.0, vmax=1)
    axes[5].set_title("Combined Score")
    axes[5].set_xlabel("Intervals")
    axes[5].set_ylabel("")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.subplots_adjust(hspace=0.5)
    plt.subplots_adjust(wspace=0.2)

    pdf_pages.savefig(fig, bbox_inches='tight')
    plt.close(fig)
    # plt.show()


<br>

### **Function - Boxplots of metrics**

* Function that generates boxplots to visualize the distribution of metrics from the results of the Mapper algorithm.

In [ ]:
def plot_boxplots(results_df, pdf_pages, title=None):

    if results_df.empty:
        print("No results were generated. Skipping boxplot generation.")
        return

    # List of metrics to be plotted
    metrics_to_plot = [
        "unclustered_samples",
        "num_clusters",
        "num_edges",
        "num_connected_components",
        "std_dev_cluster_sizes",
        "combined_score"
    ]

    # Titles for each subplot
    metric_titles = [
        "Distribution of Unclustered Samples",
        "Distribution of Number of Clusters",
        "Distribution of Number of Edges",
        "Distribution of Connected Components",
        "Distribution of Cluster Size Variation",
        "Distribution of Combined Score"
    ]

    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes = axes.flatten()
    fig.suptitle(title, fontsize=14)

    for i, metric in enumerate(metrics_to_plot):
        sns.boxplot(y=results_df[metric], ax=axes[i], width=0.4)
        axes[i].set_title(metric_titles[i], fontsize=10)
        axes[i].set_xlabel("")
        axes[i].set_ylabel("")

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.subplots_adjust(hspace=0.4)
    plt.subplots_adjust(wspace=0.4)

    pdf_pages.savefig(fig, bbox_inches='tight')
    plt.close(fig)


<br>

### **Function - Mapper graph**

* This function plots the static graph from the result of the Mapper algorithm.

  - The size of the node is proportional to its degree.

  - The color scale represents the number of samples in each cluster.

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker

def plot_graph_mapper(mapper_graph, pdf_pages, nodes_spacing=None, title=None):
    # Convert the graph to NetworkX format
    nx_graph = km.adapter.to_nx(mapper_graph)

    # Create a dictionary for the numeric node labels (e.g. 'cube1_cluster0' -> '1')
    node_labels = {node_name: str(i+1) for i, node_name in enumerate(nx_graph.nodes())}

    # Calculate the number of samples in each node. This will define the node color.
    sample_counts = [len(mapper_graph["nodes"][node_name]) for node_name in nx_graph.nodes()]

    # Calculate the degree of each node. This will define the node size.
    degrees = [d for n, d in nx_graph.degree()]

    # Create a color list for each node based on its sample count
    cmap_samples = cm.get_cmap('plasma_r')

    try:
        min_sample_counts = min(sample_counts)
        max_sample_counts = max(sample_counts)
        if min_sample_counts == max_sample_counts:
            norm_samples = mcolors.Normalize(vmin=min_sample_counts, vmax=min_sample_counts + 1)
        else:
            norm_samples = mcolors.Normalize(vmin=min_sample_counts, vmax=max_sample_counts)
        node_colors_by_sample = cmap_samples(norm_samples(sample_counts))
    except:
        print(f"  Error - Sample_counts '{sample_counts}'")


    # Calculate the median of sample counts to use as a threshold
    median_value_samples = np.median(sample_counts)

    # Create a list of font colors: white for nodes with dark colors and black for the rest
    font_colors = []
    for color in node_colors_by_sample:
        luminance = (0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2])
        font_colors.append('black' if luminance > 0.5 else 'white')

    # Create the figure and axes for the plot
    fig, ax = plt.subplots(figsize=(10, 8))

    # Define a layout for the graph to ensure consistent positioning
    # k: controls the distance between nodes. A larger value increases the spacing
    pos = nx.spring_layout(nx_graph, seed=42, k=nodes_spacing)

    # Draw the graph nodes
    nx.draw_networkx_nodes(nx_graph, pos, ax=ax,
                           node_color=node_colors_by_sample,
                           node_size=[300 + (d * 100) for d in degrees])

    # Draw the graph edges
    nx.draw_networkx_edges(nx_graph, pos, ax=ax)

    # Draw the node labels one by one to apply a specific font color to each
    for i, node_name in enumerate(nx_graph.nodes()):
        nx.draw_networkx_labels(nx_graph, pos, ax=ax,
                                labels={node_name: node_labels[node_name]},
                                font_color=font_colors[i], font_size=11, font_weight='bold')

    # Add color bar to chart
    sm = cm.ScalarMappable(cmap=cmap_samples, norm=norm_samples)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', shrink=0.75, pad=0.10)
    cbar.set_label("Number of genes", rotation=90, labelpad=-45)

    # Set color bar ticks to integers
    if max_sample_counts > min_sample_counts:
        locator = mticker.MaxNLocator(nbins='auto', integer=True)
        cbar.ax.yaxis.set_major_locator(locator)

    # Final adjustments
    plt.title(title, fontsize=10, pad=30)
    # plt.tight_layout()
    plt.subplots_adjust(left=0.05, right=0.6, top=0.55, bottom=0.05)

    pdf_pages.savefig(fig, bbox_inches='tight')
    # plt.show()
    plt.close(fig)

<br>

### **Function - Result details**

* Function to print the details of the Mapper graphs for the top results in a TXT file.

In [ ]:
def print_result_details(output_file, mapper_graph, data_df, row, index):

    # Convert the graph to NetworkX format
    nx_graph = km.adapter.to_nx(mapper_graph)

    # Create a dictionary for the numeric node labels (e.g. 'cube1_cluster0' -> '1')
    cluster_labels = {cluster_name: str(i+1) for i, cluster_name in enumerate(nx_graph.nodes())}

    # Create a list with the size of each node (number of samples)
    cluster_sizes = [len(mapper_graph["nodes"][cluster_name]) for cluster_name in nx_graph.nodes()]

    output_file.write("=" * 80 + "\n")
    output_file.write("Result #" + str(index+1) + "\n\n")

    output_file.write("Mapper algorithm parameters: \n")
    output_file.write("   Projection: " + str(row['projection_function']) + "\n" +
                      "   Clustering: " + str(row['clustering_algorithm']) + "\n" +
                      "   Intervals: " + str(row['n_cubes']) + "\n" +
                      "   Overlap: " + str(row['perc_overlap']) + "\n\n")

    output_file.write("Graph details: \n")
    output_file.write("   Number of clusters: " + str(row['num_clusters']) + "\n" +
                      "   Number of edges: " + str(row['num_edges']) + "\n" +
                      "   Connected components: " + str(row['num_connected_components'])  + "\n" +
                      "   Combined score: " + str(row['combined_score'])  + "\n\n")

    output_file.write("Samples contained in each cluster:\n")

    # List of samples from all clusters
    all_cluster_samples = []

    # Print node information
    for cluster_name, cluster_label in cluster_labels.items():
        sample_indices = mapper_graph["nodes"][cluster_name]
        sample_names = data_df.index[sample_indices].tolist()
        all_cluster_samples.extend(sample_names)
        output_file.write(f"   Cluster {cluster_label} ({len(sample_names)} samples): {', '.join(map(str, sample_names))}\n")

    # Samples not in clusters
    unique_samples = list(set(all_cluster_samples))
    samples_not_in_clusters = set(data_df.index) - set(unique_samples)
    sorted_samples_not_in_clusters = sorted(list(samples_not_in_clusters))
    if len(sorted_samples_not_in_clusters) > 0:
        output_file.write(f"   Samples not found in clusters ({len(sorted_samples_not_in_clusters)} samples): {', '.join(map(str, sorted_samples_not_in_clusters))}\n\n\n")
    else:
        output_file.write(f"   Samples not found in clusters (0 samples)\n\n\n")



<br><br><br>

---


# **4) Parameter analysis**

* The Mapper algorithm is run with different parameters.

* Plots the results as heatmaps, boxplots, and graphs.



<br>

### **Parameters to be tested**

* Specify the projection functions, clustering methods, and parameters to run the Mapper algorithm.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.cluster import AffinityPropagation
import hdbscan
from sklearn.cluster import OPTICS
from sklearn.mixture import BayesianGaussianMixture
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth


# Define the parameter ranges to test
intervals_to_test = [3, 5, 10, 20, 30]
perc_overlap_to_test = [0.01, 0.1, 0.3, 0.5]

# Define the fixed projection and clustering algorithms for the experiment
projections_to_test = [
    PCA(n_components=1, random_state=0),
    PCA(n_components=2, random_state=0),
    PCA(n_components=3, random_state=0)
]

clust_alg_to_test = [
    # Affinity propagation (AP)
    AffinityPropagation(damping=0.50, random_state=0),
    AffinityPropagation(damping=0.75, random_state=0),
    AffinityPropagation(damping=0.99, random_state=0),

    # DBSCAN
    DBSCAN(eps=0.5),
    DBSCAN(eps=3.0),
    DBSCAN(eps=5.0),
    DBSCAN(eps=10.0),

    # HDBSCAN
    hdbscan.HDBSCAN(min_cluster_size=2, min_samples=None),
    hdbscan.HDBSCAN(min_cluster_size=5, min_samples=None),
    hdbscan.HDBSCAN(min_cluster_size=8, min_samples=None),

    # MeanShift
    MeanShift(cluster_all=False),
    MeanShift(cluster_all=True),

    # OPTICS
    OPTICS(min_samples=2, max_eps=np.inf),
    OPTICS(min_samples=5, max_eps=np.inf),
    OPTICS(min_samples=8, max_eps=np.inf),

    # Variational Bayesian Gaussian Mixture (VBGMM)
    BayesianGaussianMixture(n_components=1, random_state=0),
    BayesianGaussianMixture(n_components=2, random_state=0)
]

total_parameter_sets = len(projections_to_test) * len(clust_alg_to_test) * len(intervals_to_test) * len(perc_overlap_to_test)
print(f"Total number of parameter sets to test: {total_parameter_sets}")


Total number of parameter sets to test: 1020


<br>

In [ ]:
clust_alg_labels = [
    # Affinity propagation (AP)
    "AP (damping=0.5)",
    "AP (damping=0.75)",
    "AP (damping=0.99)",

    # DBSCAN
    "DBSCAN (eps=0.5)",
    "DBSCAN (eps=3.0)",
    "DBSCAN (eps=5.0)",
    "DBSCAN (eps=10.0)",

    # HDBSCAN
    "HDBSCAN (min_cluster=2)",
    "HDBSCAN (min_cluster=5)",
    "HDBSCAN (min_cluster=8)",

    # MeanShift
    "MeanShift (cluster_all=False)",
    "MeanShift (cluster_all=True)",

    # OPTICS
    "OPTICS (min_samples=2)",
    "OPTICS (min_samples=5)",
    "OPTICS (min_samples=8)",

    # Variational Bayesian Gaussian Mixture (VBGMM)
    "VBGMM (n_components=1)",
    "VBGMM (n_components=2)"
]


<br>

### **Parameters of metrics and score**

* Specify the following parameters:

  - Range of target values ​​for the metrics.

  - Weight of each score in calculating the combined score.


In [ ]:
score_params = {
    # Target ranges [vmin, vmax]. A penalty is applied to the score only if the metric is outside this range.
    "target_unclustered": {"vmin": 0, "vmax": 5},
    "target_clusters": {"vmin": 4, "vmax": 6},
    "target_edges": {"vmin": 0, "vmax": 10},
    "target_connected_components": {"vmin": 1, "vmax": 6},
    "target_cluster_size_variation": {"vmin": 0, "vmax": 6},

    # Weight of each score in calculating the combined score
    "weight_score1": 0.4,
    "weight_score2": 0.3,
    "weight_score3": 0.1,
    "weight_score4": 0.1,
    "weight_score5": 0.1
}


<br>

### **Execution of the Mapper algorithm**

* Run the Mapper algorithm with the projection functions, clustering methods and parameters previously defined.

* Create a PDF file containing heatmaps of metrics from the results of the Mapper algorithm.

In [ ]:
# Calculate the number of samples in the input after preprocessing
total_samples = len(data_df)

# Get the total number of projection functions
total_proj_functions = len(projections_to_test)

# Paths to PDF files with graphs of metrics of all results of Mapper algorithm execution
output_pdf_heatmap_path = os.path.join(output_folder, f"All results - Heatmaps of metrics.pdf")
output_pdf_boxplot_path = os.path.join(output_folder, f"All results - Boxplots of metrics.pdf")

# List to store all results_df
all_results_list = []

# Create the multi-page PDF file using a context manager
with PdfPages(output_pdf_heatmap_path) as pdf_heatmaps, PdfPages(output_pdf_boxplot_path) as pdf_boxplots:
    for i, proj_func in enumerate(projections_to_test):
        print(f"\n\nStarting analysis with projection function [{i+1}/{total_proj_functions}]: {proj_func}")

        mapper = km.KeplerMapper(verbose=0)
        normalized_data = StandardScaler().fit_transform(data_df)
        projected_data = mapper.project(normalized_data, projection=proj_func)

        for clust_alg in tqdm(clust_alg_to_test, desc="    Progress"):
            # Perform analysis of the Mapper algorithm with different parameters
            results_df = run_mapper_analysis(mapper, normalized_data, projected_data, proj_func,
                                             clust_alg, intervals_to_test, perc_overlap_to_test)

            all_results_list.append(results_df)

            plot_title = format_plot_title(proj_func, clust_alg)
            plot_heatmaps(results_df, pdf_heatmaps, plot_title)
            plot_boxplots(results_df, pdf_boxplots, plot_title)

# Concatenate all DataFrames into a single one
all_results_df = pd.concat(all_results_list, ignore_index=True)

# Save the DataFrame with the metrics of all results as a TSV file
output_path = os.path.join(output_folder, "All results - Table of metrics.tsv")
all_results_df.to_csv(output_path, sep='\t', index=False)

print(f"\n\nComplete analysis successfully saved to: ")
print(f"   {output_pdf_heatmap_path}")
print(f"   {output_pdf_boxplot_path}")




Starting analysis with projection function [1/3]: PCA(n_components=1, random_state=0)


    Progress:   0%|          | 0/17 [00:00<?, ?it/s]



Starting analysis with projection function [2/3]: PCA(n_components=2, random_state=0)


    Progress:   0%|          | 0/17 [00:00<?, ?it/s]



Starting analysis with projection function [3/3]: PCA(n_components=3, random_state=0)


    Progress:   0%|          | 0/17 [00:00<?, ?it/s]



Complete analysis successfully saved to: 
   Output/dlbcl_integrated_mafs/All results - Heatmaps of metrics.pdf
   Output/dlbcl_integrated_mafs/All results - Boxplots of metrics.pdf


<br>

### **Graphs of combined score distributions**

* Create a PDF file containing boxplots showing the distribution of the "combined score" of each clustering method to each projection function.

* Each page focuses on a single projection function, allowing you to compare the performance of the tested clustering algorithms for each projection function.

In [ ]:
print(f"\nStarting to generate graphs of performance comparison")

# Path to the PDF with the graphs
comparison_pdf_path = os.path.join(output_folder, f"All results - Distributions of combined scores.pdf")

# Get the list of unique projection functions that were tested
unique_projections = all_results_df['projection_function'].unique()

# Create the multi-page PDF file using a context manager
with PdfPages(comparison_pdf_path) as pdf:
    for i, proj_func_str in enumerate(unique_projections):
        print(f"   Creating plot for projection [{i+1}/{len(unique_projections)}]: {proj_func_str}")

        # Filter the main DataFrame to get results for the current projection only
        df_subset = all_results_df[all_results_df['projection_function'] == proj_func_str]

        fig, ax = plt.subplots(figsize=(14, 9))

        # Create the boxplot
        sns.boxplot(x='clustering_algorithm', y='combined_score', data=df_subset, ax=ax)

        title = format_projection_title(proj_func_str)
        ax.set_title(title, fontsize=16, pad=30)


        # Configure the axes
        ax.set_xlabel("")
        ax.set_ylabel("Combined Score", fontsize=12)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.set_xticklabels(clust_alg_labels)
        plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

        # Set the y-axis limit to be between 0 and 1
        ax.set_ylim(-0.05, 1.05)

        plt.tight_layout()
        pdf.savefig(fig)
        # plt.show()
        plt.close(fig)


print(f"\nGraphs saved in: {comparison_pdf_path}")



Starting to generate graphs of performance comparison
   Creating plot for projection [1/3]: PCA(n_components=1, random_state=0)
   Creating plot for projection [2/3]: PCA(n_components=2, random_state=0)
   Creating plot for projection [3/3]: PCA(n_components=3, random_state=0)

Graphs saved in: Output/dlbcl_integrated_mafs/All results - Distributions of combined scores.pdf


<br>

### **Selection of the top results**

* Select the Mapper algorithm results that achieved the highest combined score.

In [ ]:
N = int(total_parameter_sets * 0.1)

top_n_df = all_results_df.nlargest(N, 'combined_score').reset_index(drop=True)

# Save the DataFrame with the metrics of top results as a TSV file
output_path = os.path.join(output_folder, "Top results - Table of metrics.tsv")
top_n_df.to_csv(output_path, sep='\t', index=False)

display(top_n_df)


,projection_function,clustering_algorithm,n_cubes,perc_overlap,unclustered_samples,num_clusters,num_edges,num_connected_components,std_dev_cluster_sizes,max_cluster_size,num_large_clusters,num_small_clusters,combined_score
0,"PCA(n_components=1, random_state=0)","AffinityPropagation(damping=0.99, random_state=0)",10,0.01,3,5,0,5,3.611094,11,0,1,1.000000
1,"PCA(n_components=1, random_state=0)","AffinityPropagation(damping=0.99, random_state=0)",10,0.10,1,6,2,4,5.112621,15,1,3,1.000000
2,"PCA(n_components=1, random_state=0)",MeanShift(cluster_all=False),10,0.01,5,5,0,5,5.253570,14,0,3,1.000000
3,"PCA(n_components=1, random_state=0)",MeanShift(),10,0.01,3,5,0,5,5.003998,14,0,2,1.000000
4,"PCA(n_components=1, random_state=0)",OPTICS(min_samples=2),10,0.10,1,4,2,2,5.214163,16,1,1,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,"PCA(n_components=2, random_state=0)",OPTICS(min_samples=2),5,0.50,1,7,9,2,8.258576,24,3,3,0.812357
98,"PCA(n_components=2, random_state=0)",BayesianGaussianMixture(random_state=0),5,0.50,1,7,9,2,8.258576,24,3,3,0.812357
99,"PCA(n_components=2, random_state=0)",MeanShift(cluster_all=False),5,0.50,2,7,9,2,8.692009,24,3,3,0.805133
100,"PCA(n_components=1, random_state=0)",AffinityPropagation(random_state=0),3,0.01,1,7,0,7,7.698396,23,1,6,0.801693


<br>

### **Graphs of the top results**

* Create a PDF file containing static graphs of the top results from the Mapper algorithm.

In [ ]:
# Reset mapper and normalize data
mapper = km.KeplerMapper(verbose=0)
normalized_data = StandardScaler().fit_transform(data_df)

# Path to the PDF file containing the Mapper graphs of the Top Results
output_pdf_mapper_graphs_path = os.path.join(output_folder, f"Top results - Mapper graphs.pdf")

# Path to the TXT file containing details about the Mapper graphs of the Top Results
output_txt_path = os.path.join(output_folder, f"Top results - Details of Mapper graphs.txt")

# Create the multi-page PDF file using a context manager
with PdfPages(output_pdf_mapper_graphs_path) as pdf_graphs, open(output_txt_path, 'w', encoding='utf-8') as output_file:
    print(f"\nStarting to generate graphs of the top results")
    output_file.write("Mapper graphs details of top results\n\n\n")

    for index, row in tqdm(top_n_df.iterrows(), total=len(top_n_df), desc="    Progress"):
        # Retrieve the result parameters to run the Mapper algorithm
        proj_str = row['projection_function']
        clust_str = row['clustering_algorithm']
        n_cubes = row['n_cubes']
        perc_overlap = row['perc_overlap']

        # Recreate the projection and clustering objects
        try:
            proj_func = eval(proj_str)
            clust_alg = eval(clust_str)
        except:
            print(f"  Warning: Could not recreate object for '{proj_func}' and '{clust_alg}'. Skipping execution of the Mapper algorithm.")
            continue

        # Run the Mapper algorithm with the parameters obtained from the top results
        projected_data = mapper.project(normalized_data, projection=proj_func)
        cover = km.Cover(n_cubes=n_cubes, perc_overlap=perc_overlap)
        mapper_graph = mapper.map(lens=projected_data, X=normalized_data,
                                  clusterer=clust_alg, cover=cover)

        # Plot the graph of the Mapper algorithm result in the PDF file
        plot_title = format_plot_title(proj_func, clust_alg)
        plot_title = plot_title + "\n\nIntervals: " + str(n_cubes) + " - Overlap: " + str(perc_overlap)
        plot_graph_mapper(mapper_graph, pdf_graphs, nodes_spacing=1.0, title=plot_title)

        # Prints Mapper graph details of the result in TXT file
        print_result_details(output_file, mapper_graph, data_df, row, index)


print(f"\nMapper graphs of top results saved in: {output_pdf_mapper_graphs_path}")
print(f"Mapper graphs details of top results saved in: {output_txt_path}")



Starting to generate graphs of the top results


    Progress:   0%|          | 0/102 [00:00<?, ?it/s]


Mapper graphs of top results saved in: Output/dlbcl_integrated_mafs/Top results - Mapper graphs.pdf
Mapper graphs details of top results saved in: Output/dlbcl_integrated_mafs/Top results - Details of Mapper graphs.txt


<br>

<br><br><br>

---


# **5) Exporting results**

- This section allows the user to download the Output folder, which contains the generated plots, statistics, and any other result files produced during the analysis.

- The folder is first compressed into a .zip file, then a download button is displayed.

- By clicking the button, the zipped folder will be downloaded directly to the user’s computer.

In [ ]:
import shutil
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# Function to download the zip file
def on_download_button_clicked(b):
    files.download("Output.zip")

# Compact the Output folder
shutil.make_archive("Output", 'zip', "Output")

# Create download button
download_button = widgets.Button(description="Download Output", button_style='success')
download_button.on_click(on_download_button_clicked)
display(download_button)


Button(button_style='success', description='Download Output', style=ButtonStyle())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>